# VitroVision — 01: Exploratory Data Analysis (EDA)

**วัตถุประสงค์:** สำรวจและประเมินความพร้อมของ dataset ภาพขวดเพาะเลี้ยงเนื้อเยื่อก่อนเริ่ม training

Notebook นี้ครอบคลุม:
1. ค้นหาและ parse ชื่อไฟล์ภาพทั้งหมด
2. สร้าง DataFrame สรุป metadata ของแต่ละภาพ
3. วิเคราะห์ class balance
4. กระจาย dimension และ aspect ratio
5. ตรวจหาภาพซ้ำ (duplicate detection)
6. แสดงตัวอย่างภาพแต่ละ class
7. ความครอบคลุมตาม day_point และ bottle_id
8. สรุปผลและ checklist ความพร้อมสำหรับ training

> **หมายเหตุ:** Notebook นี้ทำงานได้บน dataset เปล่า — ถ้าไม่พบภาพจะแสดงข้อความแจ้งและข้ามขั้นตอนที่เกี่ยวข้อง

## 1 — Imports & Path Setup

In [ ]:
import os, sys, warnings, hashlib
from pathlib import Path
from collections import Counter

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns

warnings.filterwarnings('ignore')

# ── ตำแหน่ง project root (notebooks/ อยู่ 1 ระดับใต้ root) ──────────────────
ROOT            = Path('..').resolve()
DATA_RAW        = ROOT / 'data' / 'raw'               # ภาพต้นฉบับ
DATA_SHELF      = ROOT / 'shelf_manager' / 'data'     # ภาพจาก VitroShelf
RESULTS_DIR     = ROOT / 'results'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# ── Labels ตรงกับ database.py STATUS_CHOICES ──────────────────────────────────
LABELS          = ['healthy', 'contaminated', 'dead']
MIN_SAMPLES_PER_CLASS = 30    # เกณฑ์ขั้นต่ำก่อน train

# ── Style ──────────────────────────────────────────────────────────────────────
LABEL_COLORS    = {'healthy': '#4CAF50', 'contaminated': '#F44336', 'dead': '#9E9E9E'}
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 100

print(f'ROOT      : {ROOT}')
print(f'DATA_RAW  : {DATA_RAW}')
print(f'DATA_SHELF: {DATA_SHELF}')

## 2 — ค้นหาและ Parse ชื่อไฟล์ภาพ

รูปแบบชื่อไฟล์ที่คาดหวัง: `{bottle_id}_day{NNN}_{seq}_{label}.jpg`  
ตัวอย่าง: `S01-A-01_day007_0001_healthy.jpg`

In [ ]:
def parse_filename(path):
    """แยก metadata จากชื่อไฟล์ตาม convention: {bottle_id}_day{NNN}_{seq}_{label}.jpg
    คืน dict หรือ None ถ้าชื่อไฟล์ไม่ตรง format"""
    stem = path.stem   # ตัด .jpg ออก
    parts = stem.split('_')
    # ต้องมีอย่างน้อย 4 ส่วน: bottle_id, day{NNN}, seq, label
    if len(parts) < 4:
        return None
    label = parts[-1]
    seq   = parts[-2]
    day_str = parts[-3]   # เช่น 'day007'
    # bottle_id อาจมี underscore ใน prefix ดังนั้นรวมส่วนที่เหลือ
    bottle_id = '_'.join(parts[:-3])
    if not day_str.startswith('day'):
        return None
    try:
        day = int(day_str[3:])
    except ValueError:
        return None
    if label not in LABELS:
        return None
    return {
        'path'      : str(path),
        'bottle_id' : bottle_id,
        'day'       : day,
        'seq'       : seq,
        'label'     : label,
    }


def discover_images(*dirs):
    """ค้นหา *.jpg แบบ recursive ใน directories ที่ระบุ"""
    records = []
    seen_paths = set()
    for d in dirs:
        d = Path(d)
        if not d.exists():
            print(f'  [ข้าม] ไม่พบ directory: {d}')
            continue
        jpgs = list(d.rglob('*.jpg'))
        print(f'  {d.relative_to(ROOT)}: พบ {len(jpgs)} ไฟล์ .jpg')
        for p in jpgs:
            if str(p) in seen_paths:
                continue
            seen_paths.add(str(p))
            rec = parse_filename(p)
            if rec is not None:
                records.append(rec)
            else:
                # ไฟล์ชื่อไม่ตรง format — เก็บไว้เตือนทีหลัง
                records.append({
                    'path'      : str(p),
                    'bottle_id' : None,
                    'day'       : None,
                    'seq'       : None,
                    'label'     : 'UNKNOWN_FORMAT',
                })
    return records


print('กำลังค้นหาภาพ...')
raw_records = discover_images(DATA_RAW, DATA_SHELF)
print(f'\nพบทั้งหมด: {len(raw_records)} ไฟล์')

## 3 — สร้าง DataFrame หลัก

เพิ่ม metadata ของภาพแต่ละไฟล์: ขนาด, aspect ratio, filesize

In [ ]:
def get_image_meta(path_str):
    """อ่าน width, height, filesize ของภาพ — คืน None ถ้าเปิดไม่ได้"""
    try:
        size_bytes = os.path.getsize(path_str)
        img = cv2.imread(path_str)
        if img is None:
            return None, None, size_bytes, True   # corrupt
        h, w = img.shape[:2]
        return w, h, size_bytes, False
    except Exception:
        return None, None, 0, True


# ── สร้าง DataFrame ────────────────────────────────────────────────────────────
if len(raw_records) == 0:
    print('ยังไม่มีภาพใน dataset')
    print('กรุณาเพิ่มภาพใน data/raw/ หรือ shelf_manager/data/ ก่อน')
    df = pd.DataFrame(columns=['path','bottle_id','day','seq','label',
                                'width','height','filesize','aspect_ratio',
                                'is_corrupt','file_hash'])
else:
    print(f'กำลังอ่าน metadata ของ {len(raw_records)} ภาพ...')
    rows = []
    for rec in raw_records:
        w, h, sz, corrupt = get_image_meta(rec['path'])
        rows.append({
            **rec,
            'width'       : w,
            'height'      : h,
            'filesize'    : sz,
            'aspect_ratio': round(w / h, 3) if (w and h) else None,
            'is_corrupt'  : corrupt,
            'file_hash'   : None,   # จะคำนวณทีหลังในหัวข้อ duplicate
        })
    df = pd.DataFrame(rows)
    print(f'DataFrame: {len(df)} rows × {len(df.columns)} cols')

# ── แสดง format ที่ไม่ตรง ──────────────────────────────────────────────────────
bad_format = df[df['label'] == 'UNKNOWN_FORMAT'] if len(df) > 0 else pd.DataFrame()
if len(bad_format) > 0:
    print(f'\n⚠️  ชื่อไฟล์ไม่ตรง format: {len(bad_format)} ไฟล์')
    print(bad_format['path'].head(5).to_string(index=False))

# กรองเฉพาะภาพที่ label ถูกต้อง
df_valid = df[df['label'].isin(LABELS)].copy()
print(f'\nภาพที่ใช้งานได้ (label ถูกต้อง): {len(df_valid)}')

if len(df_valid) > 0:
    display(df_valid.head(5))

## 4 — Class Balance

ตรวจสอบว่าแต่ละ class มีจำนวนภาพสมดุลและมากพอสำหรับ training

In [ ]:
if len(df_valid) == 0:
    print('ยังไม่มีภาพ — ข้ามการวิเคราะห์ class balance')
else:
    # นับจำนวนต่อ class
    counts = df_valid['label'].value_counts().reindex(LABELS, fill_value=0)
    total  = counts.sum()

    print('จำนวนภาพต่อ class:')
    for cls, cnt in counts.items():
        pct    = cnt / total * 100 if total > 0 else 0
        status = '✓' if cnt >= MIN_SAMPLES_PER_CLASS else '✗'
        print(f'  {status}  {cls:15s}: {cnt:5d} ภาพ  ({pct:.1f}%)')

    print(f'\nรวม: {total} ภาพ')

    # Bar chart
    fig, ax = plt.subplots(figsize=(7, 4))
    bars = ax.bar(
        counts.index,
        counts.values,
        color=[LABEL_COLORS.get(l, '#888') for l in counts.index],
        edgecolor='white',
        linewidth=1.2,
    )
    # เส้นขั้นต่ำ
    ax.axhline(MIN_SAMPLES_PER_CLASS, color='tomato', linestyle='--',
               linewidth=1.5, label=f'เกณฑ์ขั้นต่ำ ({MIN_SAMPLES_PER_CLASS})')
    for bar, cnt in zip(bars, counts.values):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.5,
                str(cnt), ha='center', va='bottom', fontsize=11, fontweight='bold')
    ax.set_title('Class Distribution', fontsize=13)
    ax.set_xlabel('Label')
    ax.set_ylabel('จำนวนภาพ')
    ax.legend()
    plt.tight_layout()
    plt.savefig(RESULTS_DIR / 'eda_class_balance.png', dpi=120)
    plt.show()

    # คำเตือน imbalance
    low_classes = [c for c, n in counts.items() if n < MIN_SAMPLES_PER_CLASS]
    if low_classes:
        print(f'\n⚠️  class ที่ยังมีภาพน้อยกว่า {MIN_SAMPLES_PER_CLASS}: {low_classes}')
        print('   แนะนำเก็บภาพเพิ่มก่อนเริ่ม training')

    max_cnt  = counts.max()
    min_cnt  = counts[counts > 0].min() if (counts > 0).any() else 1
    ratio    = max_cnt / min_cnt
    if ratio > 3:
        print(f'\n⚠️  Class imbalance ratio สูง: {ratio:.1f}x')
        print('   พิจารณาใช้ class_weight หรือ oversampling ตอน training')

## 5 — Image Dimensions & Aspect Ratio

กระจาย width, height, และ aspect ratio — ตรวจจับภาพที่เล็กเกินไปหรือผิดรูปทรง

In [ ]:
MIN_DIM = 64   # pixel — ต่ำกว่านี้ถือว่าเล็กเกินไป

if len(df_valid) == 0:
    print('ยังไม่มีภาพ — ข้ามการวิเคราะห์ dimension')
else:
    # ภาพ corrupt
    corrupt = df_valid[df_valid['is_corrupt'] == True]
    if len(corrupt) > 0:
        print(f'⚠️  พบภาพ corrupt หรือเปิดไม่ได้: {len(corrupt)} ไฟล์')
        print(corrupt['path'].head(5).to_string(index=False))

    df_ok = df_valid[df_valid['is_corrupt'] == False].copy()

    if len(df_ok) == 0:
        print('ไม่มีภาพที่อ่านได้ — ข้ามการวิเคราะห์ dimension')
    else:
        print(f'ภาพที่อ่านได้: {len(df_ok)}')
        print(f"Width   — min:{df_ok['width'].min()}  max:{df_ok['width'].max()}  mean:{df_ok['width'].mean():.0f}")
        print(f"Height  — min:{df_ok['height'].min()}  max:{df_ok['height'].max()}  mean:{df_ok['height'].mean():.0f}")
        print(f"Aspect  — min:{df_ok['aspect_ratio'].min():.3f}  max:{df_ok['aspect_ratio'].max():.3f}  mean:{df_ok['aspect_ratio'].mean():.3f}")

        # ตรวจภาพเล็ก
        tiny = df_ok[(df_ok['width'] < MIN_DIM) | (df_ok['height'] < MIN_DIM)]
        if len(tiny) > 0:
            print(f'\n⚠️  ภาพที่เล็กกว่า {MIN_DIM}px: {len(tiny)} ไฟล์')

        fig, axes = plt.subplots(1, 3, figsize=(14, 4))

        # Width distribution
        axes[0].hist(df_ok['width'], bins=30, color='steelblue', edgecolor='white')
        axes[0].axvline(MIN_DIM, color='tomato', linestyle='--', label=f'min={MIN_DIM}')
        axes[0].set_title('Width (px)'); axes[0].set_xlabel('pixels'); axes[0].legend()

        # Height distribution
        axes[1].hist(df_ok['height'], bins=30, color='mediumpurple', edgecolor='white')
        axes[1].axvline(MIN_DIM, color='tomato', linestyle='--', label=f'min={MIN_DIM}')
        axes[1].set_title('Height (px)'); axes[1].set_xlabel('pixels'); axes[1].legend()

        # Aspect ratio
        axes[2].hist(df_ok['aspect_ratio'], bins=30, color='darkorange', edgecolor='white')
        axes[2].axvline(1.0, color='green', linestyle='--', label='square (1.0)')
        axes[2].set_title('Aspect Ratio (W/H)'); axes[2].set_xlabel('ratio'); axes[2].legend()

        plt.suptitle('Image Dimension Distribution', fontsize=13)
        plt.tight_layout()
        plt.savefig(RESULTS_DIR / 'eda_dimensions.png', dpi=120)
        plt.show()

## 6 — Duplicate Detection

คำนวณ MD5 hash ของแต่ละไฟล์เพื่อตรวจว่ามีภาพซ้ำกันหรือไม่

In [ ]:
def file_md5(path_str, chunk=65536):
    """คำนวณ MD5 hash ของไฟล์"""
    h = hashlib.md5()
    try:
        with open(path_str, 'rb') as f:
            while True:
                block = f.read(chunk)
                if not block:
                    break
                h.update(block)
        return h.hexdigest()
    except Exception:
        return None


if len(df_valid) == 0:
    print('ยังไม่มีภาพ — ข้ามการตรวจ duplicate')
else:
    print(f'กำลังคำนวณ hash ของ {len(df_valid)} ไฟล์...')
    df_valid['file_hash'] = df_valid['path'].apply(file_md5)

    # นับ hash ที่ซ้ำ
    hash_counts = df_valid['file_hash'].value_counts()
    dup_hashes  = hash_counts[hash_counts > 1]

    if len(dup_hashes) == 0:
        print('ไม่พบภาพซ้ำ (MD5)')
    else:
        dup_count = df_valid[df_valid['file_hash'].isin(dup_hashes.index)].shape[0]
        print(f'⚠️  พบภาพซ้ำ: {len(dup_hashes)} กลุ่ม รวม {dup_count} ไฟล์')
        print('\nตัวอย่างภาพซ้ำ:')
        for h in dup_hashes.index[:3]:
            group = df_valid[df_valid['file_hash'] == h][['path', 'label']]
            print(group.to_string(index=False))
            print()

## 7 — Sample Grid ต่อ Class

แสดงตัวอย่างภาพ (สูงสุด 4 ภาพ) ต่อแต่ละ class เพื่อตรวจสอบ quality

In [ ]:
SAMPLES_PER_CLASS = 4

if len(df_valid) == 0:
    print('ยังไม่มีภาพ — ข้ามการแสดง sample grid')
else:
    df_ok = df_valid[df_valid['is_corrupt'] == False]

    if len(df_ok) == 0:
        print('ไม่มีภาพที่อ่านได้ — ข้ามการแสดง sample grid')
    else:
        fig = plt.figure(figsize=(14, 4 * len(LABELS)))
        gs  = gridspec.GridSpec(len(LABELS), SAMPLES_PER_CLASS, figure=fig,
                                hspace=0.4, wspace=0.05)
        has_any = False

        for row_idx, cls in enumerate(LABELS):
            cls_df  = df_ok[df_ok['label'] == cls]
            samples = cls_df.sample(min(SAMPLES_PER_CLASS, len(cls_df)),
                                    random_state=42) if len(cls_df) > 0 else cls_df

            for col_idx in range(SAMPLES_PER_CLASS):
                ax = fig.add_subplot(gs[row_idx, col_idx])
                ax.axis('off')
                if col_idx < len(samples):
                    has_any = True
                    path = samples.iloc[col_idx]['path']
                    img  = cv2.imread(path)
                    if img is not None:
                        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
                        ax.imshow(img)
                        ax.set_title(
                            Path(path).name[:28],
                            fontsize=6.5, color='#333'
                        )
                if col_idx == 0:
                    # label บน axis แรกของแต่ละแถว
                    ax.set_ylabel(cls,
                                  fontsize=11, fontweight='bold',
                                  color=LABEL_COLORS.get(cls, '#333'))

            if len(samples) == 0:
                print(f'  ยังไม่มีภาพ class "{cls}"')

        if has_any:
            plt.suptitle('ตัวอย่างภาพต่อ Class', fontsize=14, y=1.01)
            plt.savefig(RESULTS_DIR / 'eda_sample_grid.png', dpi=120, bbox_inches='tight')
            plt.show()
        else:
            plt.close()
            print('ยังไม่มีภาพสำหรับแสดง')

## 8 — Day/Time Coverage

วิเคราะห์ว่าภาพกระจายอยู่ในช่วง day_point ไหนบ้าง และแต่ละ bottle มีกี่ภาพ

In [ ]:
if len(df_valid) == 0:
    print('ยังไม่มีภาพ — ข้ามการวิเคราะห์ day/time coverage')
else:
    df_day = df_valid.dropna(subset=['day'])

    if len(df_day) == 0:
        print('ไม่สามารถอ่าน day_point ได้ — ข้ามการวิเคราะห์')
    else:
        fig, axes = plt.subplots(1, 2, figsize=(14, 4))

        # ── ภาพต่อ day_point ───────────────────────────────────────────────────
        day_counts = df_day.groupby(['day', 'label']).size().unstack(fill_value=0)
        # เพิ่ม column ที่ขาดหาย
        for cls in LABELS:
            if cls not in day_counts.columns:
                day_counts[cls] = 0
        day_counts = day_counts[LABELS]

        day_counts.plot(
            kind='bar', stacked=True, ax=axes[0],
            color=[LABEL_COLORS[l] for l in LABELS],
            edgecolor='white',
        )
        axes[0].set_title('จำนวนภาพต่อ Day Point')
        axes[0].set_xlabel('Day'); axes[0].set_ylabel('จำนวนภาพ')
        axes[0].legend(title='label', fontsize=8)
        axes[0].tick_params(axis='x', rotation=45)

        # ── ภาพต่อ bottle ─────────────────────────────────────────────────────
        bottle_counts = df_valid.groupby('bottle_id').size().sort_values(ascending=False)
        axes[1].bar(range(len(bottle_counts)), bottle_counts.values,
                    color='steelblue', edgecolor='white')
        axes[1].set_title('จำนวนภาพต่อ Bottle')
        axes[1].set_xlabel('Bottle (sorted by count)')
        axes[1].set_ylabel('จำนวนภาพ')

        plt.suptitle('Day & Bottle Coverage', fontsize=13)
        plt.tight_layout()
        plt.savefig(RESULTS_DIR / 'eda_coverage.png', dpi=120)
        plt.show()

        print(f'Day points ที่พบ: {sorted(df_day["day"].unique())}')
        print(f'Bottles ที่มีภาพ: {df_valid["bottle_id"].nunique()}')
        print(f'Bottles เฉลี่ยมี: {bottle_counts.mean():.1f} ภาพ/ขวด')

## 9 — Pixel Statistics per Class

เปรียบเทียบค่าเฉลี่ย brightness ของแต่ละ class — ช่วยตรวจสอบว่า class แยกแยะได้จากภาพ

In [ ]:
MAX_SAMPLES_STAT = 20   # จำกัดเพื่อความเร็ว

if len(df_valid) == 0:
    print('ยังไม่มีภาพ — ข้ามการวิเคราะห์ pixel statistics')
else:
    df_ok   = df_valid[df_valid['is_corrupt'] == False]
    results = []

    for cls in LABELS:
        cls_df  = df_ok[df_ok['label'] == cls]
        samples = cls_df.sample(min(MAX_SAMPLES_STAT, len(cls_df)), random_state=42)
        for _, row in samples.iterrows():
            img = cv2.imread(row['path'])
            if img is None:
                continue
            img_rgb     = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            gray        = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
            hsv         = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
            results.append({
                'label'          : cls,
                'mean_brightness': float(gray.mean()),
                'mean_green_ch'  : float(img_rgb[:, :, 1].mean()),   # channel G
                'mean_saturation': float(hsv[:, :, 1].mean()),
            })

    if len(results) == 0:
        print('ไม่สามารถอ่านภาพได้ — ข้ามการวิเคราะห์')
    else:
        df_stat = pd.DataFrame(results)

        fig, axes = plt.subplots(1, 3, figsize=(13, 4))
        metrics   = ['mean_brightness', 'mean_green_ch', 'mean_saturation']
        titles    = ['Mean Brightness', 'Mean Green Channel', 'Mean Saturation (HSV)']

        for ax, metric, title in zip(axes, metrics, titles):
            sns.boxplot(
                data=df_stat, x='label', y=metric, ax=ax,
                palette=LABEL_COLORS, order=LABELS,
            )
            ax.set_title(title); ax.set_xlabel(''); ax.set_ylabel('')

        plt.suptitle('Pixel Statistics per Class', fontsize=13)
        plt.tight_layout()
        plt.savefig(RESULTS_DIR / 'eda_pixel_stats.png', dpi=120)
        plt.show()

        print(df_stat.groupby('label')[metrics].mean().round(2))

## 10 — สรุปผลและ Readiness Checklist สำหรับ Training

In [ ]:
print('=' * 60)
print('VitroVision EDA — สรุปผลการสำรวจ Dataset')
print('=' * 60)

if len(df_valid) == 0:
    print('\n📋 สถานะ: ยังไม่มีภาพใน dataset')
    print('\nสิ่งที่ต้องทำก่อนเริ่ม training:')
    print('  1. เพิ่มภาพใน data/raw/ ตาม format: {bottle_id}_day{NNN}_{seq}_{label}.jpg')
    print('     ตัวอย่าง: S01-A-01_day007_0001_healthy.jpg')
    print('  2. หรือ sync ภาพจาก VitroShelf ไปที่ shelf_manager/data/')
    print(f'  3. เก็บให้ครบอย่างน้อย {MIN_SAMPLES_PER_CLASS} ภาพต่อ class ก่อนเริ่ม train')
    print('\n⚠️  Notebook ทำงานสมบูรณ์ — รอ dataset')
else:
    counts      = df_valid['label'].value_counts().reindex(LABELS, fill_value=0)
    df_ok       = df_valid[df_valid['is_corrupt'] == False]
    corrupt_cnt = len(df_valid) - len(df_ok)
    dup_cnt     = 0
    if 'file_hash' in df_valid.columns and df_valid['file_hash'].notna().any():
        hcounts  = df_valid['file_hash'].value_counts()
        dup_cnt  = df_valid[df_valid['file_hash'].isin(hcounts[hcounts > 1].index)].shape[0]

    print(f'\nจำนวนภาพทั้งหมด  : {len(df_valid)}')
    for cls in LABELS:
        print(f'  {cls:15s}: {counts[cls]:5d} ภาพ')
    print(f'ภาพ corrupt       : {corrupt_cnt}')
    print(f'ภาพซ้ำ (MD5)      : {dup_cnt}')
    print(f'Bottles ที่มีภาพ  : {df_valid["bottle_id"].nunique()}')
    if df_valid['day'].notna().any():
        print(f'Day range         : day{int(df_valid["day"].min()):03d} → day{int(df_valid["day"].max()):03d}')

    print('\n── Readiness Checklist ──')
    checks = [
        ('ภาพมากกว่า 0',
         len(df_valid) > 0),
        (f'ทุก class มี ≥{MIN_SAMPLES_PER_CLASS} ภาพ',
         all(counts[c] >= MIN_SAMPLES_PER_CLASS for c in LABELS)),
        ('ไม่มีภาพ corrupt',
         corrupt_cnt == 0),
        ('ไม่มีภาพซ้ำ',
         dup_cnt == 0),
        ('มีภาพครบ 3 classes',
         all(counts[c] > 0 for c in LABELS)),
    ]
    all_pass = True
    for desc, ok in checks:
        mark = '✓' if ok else '✗'
        if not ok:
            all_pass = False
        print(f'  [{mark}] {desc}')

    print()
    if all_pass:
        print('Dataset พร้อมสำหรับ training — เปิด 02_train.ipynb ได้เลย')
    else:
        print('ยังมีรายการที่ต้องแก้ไข — ตรวจสอบ [✗] ด้านบนก่อน train')

    print(f'\nไฟล์ผลลัพธ์บันทึกไว้ที่: {RESULTS_DIR}')

print('=' * 60)